## 04 — Race Strategy and Pit Stop Modeling

This notebook extends the project from pre-race forecasting into in-race tactical modeling.

The goal is to evaluate how pit stop execution, lap-time behavior, sprint context, and weather interact with pre-race expectations to influence final performance.

At the center of this notebook is one core question:

**How much can in-race strategy shift outcomes relative to pre-race expectations?**

To answer that, this notebook explores multiple tactical targets, including:
- finishing position improvement relative to grid
- whether a driver outperformed qualifying expectation
- deviations from model-based expected finish
- tactical indicators such as pit stop efficiency and pace relative to the field

Because the notebook evaluates multiple related targets, it is designed as a reusable target-driven modeling framework.

This allows the same core workflow — data preparation, model fitting, evaluation, interpretability, and visualization — to be applied consistently across several tactical outcome definitions.

## targets:

***Finish improvement relative to grid***

- finish_improvement = grid_clean - finish_position

***Outperform qualifying expectation***

- outperform_quali = (finish_position < qualifying_position).astype(int)

***Finish vs Expected Finish (Residual-Based Target)***

- expected_finish = model_from_03_prediction
- finish_residual = expected_finish - actual_finish

***Overperformance vs Expected (Binary)***

- overperformed_model = (actual_finish < expected_finish).astype(int)

***Pit Stop Efficiency Score (Derived Target)***

- pit_efficiency = avg_pit_stop_ms - field_avg_pit_stop_ms
- pit_efficiency_z = (avg_pit_stop_ms - mean) / std

***“Slow Pit Stop Penalty” Target***

- slow_pit_flag = (avg_pit_stop_ms > threshold).astype(int)
- impact_on_outcome ~ slow_pit_flag

***Position Loss After Pit Stops***

- positions_lost_post_pit = position_before_pit - position_after_pit

***Lap Time Consistency***

- lap_time_std
- consistency_score = 1 / std_lap_time

***Pace Relative to Field***

- pace_delta = avg_lap_time - race_avg_lap_time

***Fastest Lap Indicator***

- fastest_lap_flag = (fastest_lap_rank == 1).astype(int)

***Wet Race Overperformance***

- wet_overperformance = (finish_improvement > 0) & (is_wet_race == 1)

***High Altitude Performance Shift***

- high_altitude_performance = finish_improvement if high_altitude_track == 1

***Strategy Gain vs Expected Gain***

- expected_improvement = function_of_grid
- strategy_gain = actual_improvement - expected_improvement

***Monte Carlo Outcome Deviation***

- simulated_expected_finish
- actual_finish - simulated_finish

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score, roc_auc_score
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from xgboost import XGBRegressor, XGBClassifier

import shap

## Part 1 — Setup and target registry

Define a dictionary that stores metadata for each target.

In [ ]:
TARGET_SPECS = {
    "finish_improvement": {
        "type": "regression",
        "label": "Finish Improvement Relative to Grid",
        "target_col": "finish_improvement",
        "metrics": ["rmse", "mae", "r2"]
    },
    "outperform_quali": {
        "type": "classification",
        "label": "Outperform Qualifying Expectation",
        "target_col": "outperform_quali",
        "metrics": ["roc_auc", "calibration", "lift"]
    },
    "finish_residual": {
        "type": "regression",
        "label": "Finish vs Expected Finish Residual",
        "target_col": "finish_residual",
        "metrics": ["rmse", "mae", "r2"]
    },
    "overperformed_model": {
        "type": "classification",
        "label": "Overperformed Model Expectation",
        "target_col": "overperformed_model",
        "metrics": ["roc_auc", "calibration", "lift"]
    }
}

## Part 2 — Shared feature groups

Define reusable feature families.

In [ ]:
FEATURE_GROUPS = {
    "tactical_core": [...],
    "contextual": [...],
    "pre_race_controls": [...],
    "weather": [...],
    "sprint": [...]
}

## Part 3 — Shared preprocessing function

function to:

- select columns
- coerce numeric types
- drop missing rows
- perform temporal split

In [ ]:
def prepare_model_data(df, target_col, feature_cols, split_year=2021):
    model_df = df[[target_col, "year"] + feature_cols].copy()

    for col in model_df.columns:
        model_df[col] = pd.to_numeric(model_df[col], errors="coerce")

    model_df = model_df.dropna().copy()

    train = model_df[model_df["year"] <= split_year].copy()
    test = model_df[model_df["year"] > split_year].copy()

    X_train = train[feature_cols]
    y_train = train[target_col]
    X_test = test[feature_cols]
    y_test = test[target_col]

    return model_df, train, test, X_train, y_train, X_test, y_test

## Part 4 — Separate modeling functions by problem type
For regression

Create a shared function that:

- fits linear baseline
- fits XGBoost regressor
- returns metrics
- returns residuals
- returns feature importance / SHAP
- For classification

Create a shared function that:

- fits logistic regression baseline
- fits XGBoost classifier
- returns ROC-AUC
- returns calibration data
- returns lift data
- returns SHAP

Part 5 — Target loop